# Traffic Sign Classifier: Adversarial Robustness via Synthetic Data

This notebook tests whether synthetic training data makes a CNN traffic sign classifier harder to fool with adversarial attacks without sacrificing accuracy on real images.

**Setup:** Two architecturally identical CNNs are compared. One is trained on the original imbalanced GTSRB subset; the other on the same data balanced with diffusion-generated (DiT) synthetic images. Both are attacked with two white-box methods: **Carlini-Wagner (CW)** and **DeepFool**, using the Adversarial Robustness Toolbox (ART).

This notebook contains the evaluation pipeline for the GTSRB+DiT (synthetic-augmented) model: loading the trained model, measuring clean accuracy, crafting adversarial examples, and measuring robustness.

> **Note on reproducibility:** trained model weights (`.h5`) and the GTSRB+DiT image folders are not included due to size. Training cells are intentionally left commented out; the notebook loads pre-trained weights and pre-computed adversarial images. See the README for how to obtain the data.


## 1. Setup and Data Loading

In [ ]:
#Import libraries and dependencies
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Conv2D, Dropout, MaxPool2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import L2
from sklearn.metrics import accuracy_score
from tensorflow.keras.models  import load_model
from art.attacks.evasion import CarliniL2Method
from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import DeepFool
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os, time, datetime, pickle, cv2, glob
import tqdm as notebook_tqdm

In [ ]:
#Check GPU
if tf.test.gpu_device_name() != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(tf.test.gpu_device_name()))

!nvidia-smi -L

In [ ]:
#Specify the path to training folder and generating the label of training data
train_dir = './gtsrb+dit/Train/'
classes = [str(x) for x in range(8)]

In [ ]:
#Generate training and validation data with image generator
def image_generator(train_parent_directory):
    train_datagen = ImageDataGenerator(rescale=1/255, validation_split = 0.2)
    return train_datagen.flow_from_directory(train_parent_directory, target_size = (30,30), batch_size = 100, class_mode = 'categorical', classes = classes, subset='training'), train_datagen.flow_from_directory(train_parent_directory, target_size=(30,30), batch_size = 25, class_mode = 'categorical', classes = classes, subset='validation')

train_generator, val_generator = image_generator(train_dir)

In [ ]:
x_train, y_train = next(train_generator)
x_val, y_val  = next(val_generator)

In [ ]:
#Visualize few examples of training data
f, axs = plt.subplots(1, 12, figsize=(15, 4))
for j in range(len(axs)):
    axs[j].imshow(x_train[j], cmap='binary')
    axs[j].axis('off')

In [ ]:
#visualize distribution of classes

image_counts_per_class = {class_id: 0 for class_id in classes}

# Iterate over all class directories to count the images
for class_id in classes:
    class_dir_path = os.path.join(train_dir, class_id)
    if os.path.isdir(class_dir_path):
        # Count only image files
        image_counts_per_class[class_id] = len([name for name in os.listdir(class_dir_path) if os.path.isfile(os.path.join(class_dir_path, name))])

# Plot the distribution
class_ids = list(image_counts_per_class.keys())
image_counts = list(image_counts_per_class.values())

# Plotting
plt.figure(figsize=(15, 7))
plt.bar(class_ids, image_counts, color='blue')
plt.title('Number in each class of GTSRB dataset')
plt.xlabel('Class ID')
plt.ylabel('Number of images')
plt.xticks(rotation=90)  # Rotates X-Axis Ticks
plt.tight_layout()  # Adjust layout to prevent clipping of tick-labels
plt.show()

In [ ]:
#Generate Test Data
y_test=pd.read_csv("./gtsrb+dit/Test-dit.csv")
labels=y_test['Path'].to_numpy()
y_test=y_test['ClassId'].values

data=[np.array(image.img_to_array(image.load_img('./gtsrb+dit/Test/'+f.replace('Test/', ''), target_size=(30, 30)))) for f in labels]

X_test=np.array(data)/255

## 2. Model Definition and Loading

The **Standard CNN** architecture (from Princeton's DARTS work): convolutional layers with dropout for regularization, max pooling, and dense layers. Output is 8 classes (the 8 sign types tested) rather than the full 43. Training is commented out below (we load the pre-trained weights directly).

In [ ]:
#Define the model
def build_cnn():
    l2_reg = L2(0.001)
    inpt = Input(shape=(30,30,3))
    x = Dropout(rate=0.1)(Conv2D(32, (5, 5), padding='same', activation='relu')(inpt))
    x = MaxPool2D(pool_size=(2, 2))(Dropout(rate=0.2)(Conv2D(32, (5, 5), padding='same', activation='relu')(x)))
    x = MaxPool2D(pool_size=(2, 2))(Dropout(rate=0.3)(Conv2D(64, (5, 5), padding='same', activation='relu')(Dropout(rate=0.3)(Conv2D(64, (5, 5), padding='same', activation='relu')(x)))))
    x = Dropout(rate=0.5)(Dense(200, activation='relu', kernel_regularizer=l2_reg)(Flatten()(x)))
    output = Dense(8, activation=None)(Dropout(rate=0.5)(Dense(200, activation='relu', kernel_regularizer=l2_reg)(x)))  # Changed from 43 to 8
    model = Model(inputs=inpt, outputs=output)
    model.compile(optimizer=Adam(learning_rate=ExponentialDecay(initial_learning_rate=1e-3, decay_steps=20000, decay_rate=0.9), beta_1=0.9, beta_2=0.999, epsilon=1e-08), loss=CategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
    return model

In [ ]:
model_cnn = build_cnn()
model_cnn.summary()

In [ ]:
# ##training
# history_cnn = model_cnn.fit(
#       train_generator,
#       validation_data = val_generator,
#       epochs=10,
#       verbose=1)

In [ ]:
# # Save model
# model_cnn.save('./gtsrb+dit/gtsrb+dit.h5')

In [ ]:
model_cnn = load_model('./gtsrb+dit/gtsrb+dit.h5')

### Clean accuracy

Baseline: how well the synthetic-augmented model reads normal, un-attacked test images.

In [ ]:
# Predict model accuracy on clean test images
pred = np.argmax(model_cnn.predict(X_test), axis=-1)
accuracy_score(y_test, pred)

## Setup Adversarial Attack

In [ ]:
#Specify Classifier with ART Library
cnn_classifier = TensorFlowV2Classifier(model=model_cnn, nb_classes=8, input_shape=(30,30,3), loss_object=CategoricalCrossentropy())

In [ ]:
#copy clean test images for perturbation
x_test_adversarial, y_test_adversarial = X_test.copy(), y_test.copy()

In [ ]:
#Measure L2 Distance Between Clean and Adversarial Images
calc_distance = lambda img1, img2: np.sum((img1-img2)**2)

In [ ]:
#Get Model Prediction on Clean Images (copy)
prediction = model_cnn.predict(x_test_adversarial)
probability_pred_clean = np.max(prediction, axis=1)
class_pred_clean = np.argmax(prediction, axis=1)

In [ ]:
#Visualize Clean and Adversarial Images and Distance
def calculateDistance(img1, img2):
    return np.sum((img1-img2)**2)

In [ ]:
def NormalizeData(data):
    return (data - np.min(data)) / (np.max(data) - np.min(data))

### Carlini-Wagner

In [ ]:
carlini_attack = CarliniL2Method(classifier = cnn_classifier, targeted=False)

In [ ]:
# # Define the checkpoint directory and final pickle file
# checkpoint_dir = './gtsrb+dit/cw_files/'
# if not os.path.exists(checkpoint_dir):
#     os.makedirs(checkpoint_dir)
# final_pickle_file = checkpoint_dir + 'cw_final.pk'

# # Load checkpoint if any exists
# start_index = 0
# checkpoint_files = sorted(glob.glob(checkpoint_dir + 'checkpoint*.pk'))
# if checkpoint_files:
#     last_checkpoint_file = checkpoint_files[-1]
#     with open(last_checkpoint_file, 'rb') as handle:
#         last_batch = pickle.load(handle)
#     start_index = int(last_checkpoint_file.split('_')[-1].split('.')[0]) * 100
# else:
#     last_batch = []

# for i in range(start_index, len(x_test_adversarial)):
#     last_batch.append(carlini_attack.generate(x=x_test_adversarial[i:i+1]))

#     # Save a checkpoint every 100 samples
#     if (i+1) % 100 == 0:
#         checkpoint_file = os.path.join(checkpoint_dir, f'checkpoint_{i//100}.pk')
#         with open(checkpoint_file, 'wb') as handle:
#             pickle.dump(last_batch, handle, protocol=pickle.HIGHEST_PROTOCOL)
#         print(f"Generated {i+1}/{len(x_test_adversarial)} adversarial samples")
#         last_batch = []

# # After the loop, save any remaining adversarial samples in last_batch
# if last_batch:  # This checks if last_batch is not empty
#     checkpoint_file = os.path.join(checkpoint_dir, f'checkpoint_{i//100 + 1}.pk')
#     with open(checkpoint_file, 'wb') as handle:
#         pickle.dump(last_batch, handle, protocol=pickle.HIGHEST_PROTOCOL)
#     print(f"Generated {i+1}/{len(x_test_adversarial)} adversarial samples - final batch")

# end = time.time()
# print(f"Generated adversarial samples in {end - start} seconds")


In [ ]:
# # Combine carlini_attack checkpoints

# checkpoint_dir = './gtsrb+dit/cw_files/'

# # Step 1: Combine all existing checkpoints
# all_adversarial_samples = []
# for checkpoint_file in sorted(glob.glob(checkpoint_dir + 'checkpoint_*.pk')):
#     with open(checkpoint_file, 'rb') as handle:
#         batch = pickle.load(handle)
#         all_adversarial_samples.extend(batch)

# # Step 3: Combine everything into one pk file
# final_pickle_file = checkpoint_dir + 'x_test_carlini.pk'
# with open(final_pickle_file, 'wb') as handle:
#     pickle.dump(all_adversarial_samples, handle, protocol=pickle.HIGHEST_PROTOCOL)

# print(f"Combined all adversarial samples into {final_pickle_file}")


In [ ]:
# Load previously generated CW images
with open(r'./gtsrb+dit/cw_files/x_test_carlini.pk', 'rb') as input_file:
  x_test_carlini = pickle.load(input_file)

x_test_carlini = np.array(x_test_carlini).squeeze(axis=1) #(3780, 30, 30, 3)

In [ ]:
# Predict model accuracy on CW test images
pred_cw = np.argmax(model_cnn.predict(x_test_carlini), axis=-1)
accuracy_score(y_test, pred_cw)

In [ ]:
#Calculate average L2 distance between clean vs CW images
sum = 0
for i in range(len(x_test_adversarial)):
  sum += calculateDistance(NormalizeData(x_test_adversarial[i]), NormalizeData(x_test_carlini[i]))

avgL2_cw = sum / len(X_test)
print("Avg L2 distance of generated CW images: ", avgL2_cw)

In [ ]:
# visualize clean versus CW
f, axs = plt.subplots(2, 10, figsize=(15, 8))
distance_cw = [calculateDistance(x1, x2) for x1, x2 in zip( NormalizeData(x_test_adversarial), NormalizeData(x_test_carlini) ) ]
titles = ['Clean Images', 'CW Adversarial Images']

# Set titles for the first column in each row
for row, title in enumerate(titles):
    axs[row, 0].set_ylabel(f"{title}\n", fontsize=16, rotation=0, ha='right')

for i in range(10):
    v = NormalizeData(x_test_adversarial[i])
    cw_img = NormalizeData(x_test_carlini[i])
    axs[0, i].imshow(v, cmap='binary')
    axs[0, i].text(15, 35, f'L2 distance: {distance_cw[i]:.2f}', ha='center', fontsize=8, color='red')
    axs[1, i].imshow(cw_img, cmap='binary')
    axs[0, i].axis('off')
    axs[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
#Get Model Prediction on Adversarial Images
prediction_cw = model_cnn.predict(x_test_carlini)

probability_pred_cw = np.max(prediction_cw, axis=1)
class_pred_cw = np.argmax(prediction_cw, axis=1)

In [ ]:
#Visualize Clean, Perturbed Added, and Adversarial Images on Successfull Attack
num_plots=10
f, axs = plt.subplots(3, num_plots, figsize=(16, 10))
title = ['Clean Images', 'Perturbation Added', 'CW Images']

for row, ax in enumerate(axs, start=0):
    ax[0].set_title("%s \n" % title[row], loc='left', fontsize=14, pad = 0)

a = 0

for i in range(len(x_test_adversarial)):
    v = NormalizeData(x_test_adversarial[i])
    if (y_test_adversarial[i] == class_pred_clean[i]) and (y_test_adversarial[i] != class_pred_cw[i]):
        if a < num_plots:
            axs[0, a].imshow(v, cmap='binary')
            axs[1, a].imshow( v - NormalizeData(x_test_carlini[i]), cmap='binary')
            axs[2, a].imshow(NormalizeData(x_test_carlini[i]), cmap='binary')
            axs[0, a].axis('off')
            axs[1, a].axis('off')
            axs[2, a].axis('off')
            a += 1

In [ ]:
#Visualize successful attacks
num_plots=7
f, axs = plt.subplots(2, num_plots, figsize=(15, 8))
title = ['Clean Images', 'CW Images']

for row, ax in enumerate(axs, start=0):
    ax[0].set_title("%s \n" % title[row], loc='left', fontsize=16)

a = 0
for i in range(len(x_test_adversarial)):
    v = NormalizeData(x_test_adversarial[i])
    if (y_test_adversarial[i] == class_pred_clean[i]) and (y_test_adversarial[i] != class_pred_cw[i]):
        if a < num_plots:
            axs[0, a].imshow(v, cmap='binary')
            axs[0, a].text(15, 32, 'True Class: '+ str(y_test_adversarial[i]), ha='center')
            axs[0, a].text(15, 35, 'Predicted Class: '+ str(class_pred_clean[i]), ha='center')
            axs[0, a].text(15, 38, 'Probability: '+ str(probability_pred_clean[i]), ha='center')
            axs[1, a].imshow(NormalizeData(x_test_carlini[i]), cmap='binary')
            axs[1, a].text(15, 32, 'True Class: '+ str(y_test_adversarial[i]), ha='center')
            axs[1, a].text(15, 35, 'Predicted Class: '+ str(class_pred_cw[i]), ha='center')
            axs[1, a].text(15, 38, 'Probability: '+ str(probability_pred_cw[i]), ha='center')
            axs[0, a].axis('off')
            axs[1, a].axis('off')
            a += 1

### DeepFool

In [ ]:
df_attack = DeepFool(classifier= cnn_classifier)

In [ ]:
# start = time.time()
# x_test_df = df_attack.generate(x=x_test_adversarial)
# end = time.time()

# print(f"Runtime for generating adv images with DeepFool is {end - start} second")
# df_dir = './gtsrb+dit/df_files/'
# if not os.path.exists(df_dir):
#     os.makedirs(df_dir)

# #Save generated images into pickle file
# with open(df_dir + 'x_test_df.pk', 'wb') as handle:
#     pickle.dump(x_test_df, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Load previously generated DF images
with open(r'./gtsrb+dit/df_files/x_test_df.pk', 'rb') as input_file:
  x_test_df = pickle.load(input_file) #(3780, 30, 30, 3)

In [ ]:
# Predict model accuracy on DeepFool test images
pred_df = np.argmax(model_cnn.predict(x_test_df), axis=-1)
accuracy_score(y_test, pred_df)

In [ ]:
#Calculate average L2 distance between clean vs DeepFool images
sum = 0
for i in range(len(x_test_adversarial)):
  sum += calculateDistance(x_test_adversarial[i], x_test_df[i])

avgL2_df = sum / len(X_test)
print("Avg L2 distance of generated DeepFool images: ", avgL2_df)

In [ ]:
# visualize clean versus DeepFool
num_plots=10
f, axs = plt.subplots(2, num_plots, figsize=(15, 8))
distance_df = [calculateDistance(x1, x2) for x1, x2 in zip( NormalizeData(x_test_adversarial), NormalizeData(x_test_df) )]
titles = ['Clean Images', 'DeepFool Adversarial Images']

# Set titles for the first column in each row
for row, title in enumerate(titles):
    axs[row, 0].set_ylabel(f"{title}\n", fontsize=16, rotation=0, ha='right')

for i in range(num_plots):
    v = NormalizeData(x_test_adversarial[i])
    df_img = NormalizeData(x_test_df[i])
    axs[0, i].imshow(v, cmap='binary')
    axs[0, i].text(15, 35, f'L2 distance: {distance_df[i]:.2f}', ha='center', fontsize=8, color='red')
    axs[1, i].imshow(df_img, cmap='binary')
    axs[0, i].axis('off')
    axs[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
#Get Model Prediction on Adversarial Images
prediction_df = model_cnn.predict(x_test_df)

probability_pred_df = np.max(prediction_df, axis=1)
class_pred_df = np.argmax(prediction_df, axis=1)

In [ ]:
#Visualize Clean, Perturbed Added, and Adversarial Images on Successful Attack
num_plots=10
f, axs = plt.subplots(3, num_plots, figsize=(16, 10))
title = ['Clean Images', 'Perturbation Added', 'DeepFool Images']

for row, ax in enumerate(axs, start=0):
    ax[0].set_title("%s \n" % title[row], loc='left', fontsize=14, pad = 0)

a = 0

for i in range(len(x_test_adversarial)):
    v = NormalizeData(x_test_adversarial[i])
    if (y_test_adversarial[i] == class_pred_clean[i]) and (y_test_adversarial[i] != class_pred_df[i]):
        if a < num_plots:
            axs[0, a].imshow(v, cmap='binary')
            axs[1, a].imshow(v-NormalizeData(x_test_df[i]), cmap='binary')
            axs[2, a].imshow(NormalizeData(x_test_df[i]), cmap='binary')
            axs[0, a].axis('off')
            axs[1, a].axis('off')
            axs[2, a].axis('off')
            a += 1

In [ ]:
#Visualize successful attacks
num_plots=7
f, axs = plt.subplots(2, num_plots, figsize=(15, 8))
title = ['Clean Images', 'DeepFool Images']

for row, ax in enumerate(axs, start=0):
    ax[0].set_title("%s \n" % title[row], loc='left', fontsize=16)

a = 0
for i in range(len(x_test_adversarial)):
    v = NormalizeData(x_test_adversarial[i])
    if (y_test_adversarial[i] == class_pred_clean[i]) and (y_test_adversarial[i] != class_pred_df[i]):
        if a < num_plots:
            axs[0, a].imshow(v, cmap='binary')
            axs[0, a].text(15, 32, 'True Class: '+ str(y_test_adversarial[i]), ha='center')
            axs[0, a].text(15, 35, 'Predicted Class: '+ str(class_pred_clean[i]), ha='center')
            axs[0, a].text(15, 38, 'Probability: '+ str(probability_pred_clean[i]), ha='center')
            axs[1, a].imshow(NormalizeData(x_test_df[i]), cmap='binary')
            axs[1, a].text(15, 32, 'True Class: '+ str(y_test_adversarial[i]), ha='center')
            axs[1, a].text(15, 35, 'Predicted Class: '+ str(class_pred_df[i]), ha='center')
            axs[1, a].text(15, 38, 'Probability: '+ str(probability_pred_df[i]), ha='center')
            axs[0, a].axis('off')
            axs[1, a].axis('off')
            a += 1